In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install --upgrade transformers


In [ ]:
!pip install datasets evaluate rouge-score

In [ ]:
import pandas as pd

# Load the dataset from the URL
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/satp_location_matched.csv"
data = pd.read_csv(url, dtype=str)

,incident_number,state,district,block,village_name,other_areas,constituency,incident_summary,Matched_Location
0,101010701,Andhra Pradesh,Hyderabad,Gachibowli (Rangareddy),NaN,Cyberabad,Serilingampally,An alleged arms supplier to the Communist Part...,"hyderabad, cyberabad"
1,101010901,Andhra Pradesh,Nizamabad,NaN,Kamareddy,NaN,Kamareddy,A Kamareddy dalam (squad) member belonging to ...,"nizamabad, kamareddy, kamareddy"
2,101030601,Andhra Pradesh,Khammam,NaN,Bhadrachalam,NaN,Bhadrachalam,Senior CPI-Maoist 'Polit Bureau' and 'central ...,"khammam, bhadrachalam, bhadrachalam"
3,101051602,Andhra Pradesh,Vishakhapatnam,NaN,NaN,Visakha Agency,Rayadurg,A TDP leader and former Sarpanch of Jerrela Gr...,"vishakhapatnam, visakha agency"
4,101060701,Andhra Pradesh,Visakhapatnam,GK Veedhi,Teegalabanda,Pedalavasa,Rayadurg,The CPI-Maoist cadres blasted coffee pulping u...,"veedhi, teegalabanda, pedavalasa"


In [ ]:
# Remove rows where 'Matched_Location' is 'Unknown'
data = data[data['Matched_Location'] != 'Unknown'].copy()

In [ ]:


# Load the dataset from the URL
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/satp_clean.csv"
tempdata = pd.read_csv(url, dtype=str)

In [ ]:
# Filter tempdata for the incident_number '1406200901'
incident_row = tempdata[tempdata['incident_number'] == '1406200901']

# Display the filtered row
display(incident_row)

,incident_number,state,district,block,village_name,other_areas,constituency,longitude,latitude,year,...,commander_arrests,cadre_arrests,sympathizer_arrests,unknown_arrests,total_surrenders,commander_surrenders,cadre_surrenders,sympathizer_surrenders,unknown_surrenders,incident_summary
8145,1406200901,Odisha,Kandhmal,Daringbadi,Sonepur,NaN,G. Udayagiri,20.848,83.895,2009,...,0,0,0,0,0,0,0,0,0,"Four persons, including the husbands of two el..."


In [ ]:
# Select only the required columns from tempdata
tempdata_coords = tempdata[['incident_number', 'latitude', 'longitude']]

# Merge with the data DataFrame
data = pd.merge(data, tempdata_coords, on='incident_number', how='left')

# Display the first few rows to verify the merge
display(data.head())

## Improved Prompt Engineering

We've updated the prompt to be more explicit about the expected output format:
- **Old**: "Extract the location of the incident: {summary}"
- **New**: "Extract location hierarchy from incident: {summary}\nFormat: state, district, block, village (comma-separated, in order). Omit missing levels."

This should help the model:
1. Understand the hierarchical structure (state → district → block → village)
2. Learn the expected format (comma-separated)
3. Handle missing levels appropriately
4. Maintain consistent ordering


In [ ]:
# Drop tempdata and tempdata_coords dataframes to free up memory
del tempdata
del tempdata_coords

In [ ]:

from datasets import Dataset, DatasetDict

def prepare_data(df):
    # Improved prompt with explicit format instructions
    inputs = [
        f"Extract location hierarchy from incident: {summary}\nFormat: state, district, block, village (comma-separated, in order). Omit missing levels."
        for summary in df['incident_summary']
    ]
    targets = df['Matched_Location'].tolist()
    return {'input_text': inputs, 'target_text': targets}

dataset = Dataset.from_dict(prepare_data(data))

# Split into train, validation, and test sets (80% train, 10% val, 10% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = dataset['test'].train_test_split(test_size=0.5, seed=42)

# Merge splits into a DatasetDict
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 7854
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 982
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 982
    })
})


In [ ]:
from transformers import T5Tokenizer

# Initialize the tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# # Tokenization function
# def preprocess_function(examples):
#     # Tokenize the input and target text
#     model_inputs = tokenizer(
#         examples['input_text'],
#         max_length=512,
#         truncation=True,
#         padding="max_length"
#     )
#     labels = tokenizer(
#         examples['target_text'],
#         max_length=64,
#         truncation=True,
#         padding="max_length"
#     ).input_ids

#     # # Replace padding token id's in the labels with -100
#     # labels = [
#     #     [(l if l != tokenizer.pad_token_id else -100) for l in label]
#     #     for label in labels
#     # ]
#     model_inputs["labels"] = labels
#     return model_inputs

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization
tokenized_datasets = dataset.map(preprocess_function, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/7854 [00:00<?, ? examples/s]

Map:   0%|          | 0/982 [00:00<?, ? examples/s]

Map:   0%|          | 0/982 [00:00<?, ? examples/s]

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Calculate exact match
    exact_matches = [
        int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)
    ]
    accuracy = sum(exact_matches) / len(exact_matches)

    return {"exact_match": accuracy * 100}  # Return as percentage


In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer
import evaluate

# Initialize the model
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.to(device)

# Define compute metrics
# rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in labels as we can't decode them
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    # rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    # Compute BLEU scores
    bleu_result = bleu.compute(predictions=decoded_preds, references=[[ref] for ref in decoded_labels])

    # Combine results
    metrics = {
        # "rouge1": rouge_result["rouge1"].mid.fmeasure * 100,
        # "rouge2": rouge_result["rouge2"].mid.fmeasure * 100,
        # "rougeL": rouge_result["rougeL"].mid.fmeasure * 100,
        "bleu": bleu_result["bleu"] * 100,
    }
    return metrics

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Output directory for checkpoints and logs
    eval_strategy="epoch",    # Evaluate at the end of every epoch
    learning_rate=5e-5,             # Learning rate
    per_device_train_batch_size=8,  # Batch size for training
    per_device_eval_batch_size=4,   # Batch size for evaluation
    num_train_epochs=3,             # Number of training epochs
    weight_decay=0.01,              # Weight decay for regularization
    save_strategy="epoch",          # Save the model at each epoch
    logging_dir="./logs",           # Directory for storing logs
    logging_steps=100,              # Log every 100 steps
    load_best_model_at_end=True, # Load the best model at the end of training
    report_to='none'
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    # compute_metrics=compute_metrics
)


In [ ]:

# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.089200,0.073098
2,0.074200,0.066961
3,0.071400,0.064558


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2946, training_loss=0.13490951619662966, metrics={'train_runtime': 3414.9844, 'train_samples_per_second': 6.9, 'train_steps_per_second': 0.863, 'total_flos': 1.434826581737472e+16, 'train_loss': 0.13490951619662966, 'epoch': 3.0})

In [ ]:
# Save the model and tokenizer
model.save_pretrained('/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model')
tokenizer.save_pretrained('/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model')


('/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/tokenizer_config.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/special_tokens_map.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/spiece.model',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/added_tokens.json')

In [ ]:
# Evaluate the model on the test set
eval_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"Test Set Evaluation Results: {eval_results}")


Test Set Evaluation Results: {'eval_loss': 0.06318316608667374, 'eval_runtime': 47.1823, 'eval_samples_per_second': 20.813, 'eval_steps_per_second': 5.214, 'epoch': 3.0}


In [ ]:
def predict_location(summary):
    # Prepare the input text with improved prompt
    input_text = f"Extract location hierarchy from incident: {summary}\nFormat: state, district, block, village (comma-separated, in order). Omit missing levels."
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).input_ids

    # Generate predictions
    outputs = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)

    # Decode the output
    predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_text

model.to('cpu')
# Example usage
incident_summary = "Three Communist Party of India-Maoist (CPI-Maoist) cadres, identified as Madvi Bhima (52), Madkam Hidma alias Sai Denga (33), and Paadam Ayate (24), surrendered in Sukma District in the Bastar division of Chhattisgarh on September 19, reports devdiscourse.com. Bhima was the ‘president’ of the Nilamadgu Revolutionary People’s Committee (RPC) Dandakaranya Adivasi Kisan Mazdoor Sangthan (DAKMS) unit of the Maoist group and carried a INR 100,000 reward. Hidma served as ‘vice-president’, while Ayate was the ‘head’ of Palachalma RPC 'Jantana Sarkar' (people's government of the Maoists), each also carrying a INR 100,000 reward. The trio cited their disillusionment with Maoist atrocities and were impressed by the state's Naxalite [Left Wing Extremist, LWE] elimination policy and welfare schemes as reasons for their surrender."
predicted_locations = predict_location(incident_summary)
print(f"Predicted Locations: {predicted_locations}")


Predicted Locations: chhattisgarh, sukma


# **LOAD MODEL**

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer
model_path = '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'Using device: {device}')


Using device: cuda
Using device: cuda


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Load the dataset from the URL
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/satp_location_matched_lat_long.csv"
data = pd.read_csv(url, dtype=str)

def prepare_data(df):
    # Improved prompt with explicit format instructions
    inputs = [
        f"Extract location hierarchy from incident: {summary}\nFormat: state, district, block, village (comma-separated, in order). Omit missing levels."
        for summary in df['incident_summary']
    ]
    targets = df['Matched_Location'].tolist()
    return {'input_text': inputs, 'target_text': targets}

dataset = Dataset.from_dict(prepare_data(data))

# Split into train, validation, and test sets (80% train, 10% val, 10% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = dataset['test'].train_test_split(test_size=0.5, seed=42)

# Merge splits into a DatasetDict
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization
tokenized_dataset_test = dataset["test"].map(preprocess_function, batched=True)
# Set format for PyTorch
tokenized_dataset_test.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)


Map:   0%|          | 0/982 [00:00<?, ? examples/s]

In [ ]:
import numpy as np

def compute_metrics(predictions, labels):
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]

    # Calculate exact match
    exact_matches = [
        int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)
    ]
    accuracy = sum(exact_matches) / len(exact_matches)

    # Calculate F1 on location tokens
    def split_into_tokens(text):
        """Split location string into individual location tokens"""
        if not text or text.strip() == '':
            return set()
        return set([token.strip().lower() for token in text.split(',') if token.strip()])

    total_precision = 0.0
    total_recall = 0.0
    valid_pairs = 0

    for pred, label in zip(decoded_preds, decoded_labels):
        pred_tokens = split_into_tokens(pred)
        label_tokens = split_into_tokens(label)
        
        if len(label_tokens) == 0:  # Skip if no ground truth tokens
            continue
            
        valid_pairs += 1
        
        # Calculate precision for this example
        if len(pred_tokens) == 0:
            precision = 0.0
        else:
            correct_tokens = pred_tokens & label_tokens
            precision = len(correct_tokens) / len(pred_tokens)
        
        # Calculate recall for this example
        correct_tokens = pred_tokens & label_tokens
        recall = len(correct_tokens) / len(label_tokens) if len(label_tokens) > 0 else 0.0
        
        total_precision += precision
        total_recall += recall
    
    # Average precision and recall across all examples
    avg_precision = total_precision / valid_pairs if valid_pairs > 0 else 0.0
    avg_recall = total_recall / valid_pairs if valid_pairs > 0 else 0.0
    
    # Calculate F1 score
    if avg_precision + avg_recall == 0:
        f1_tokens = 0.0
    else:
        f1_tokens = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall)

    return {
        "exact_match": accuracy * 100,  # Return as percentage
        "f1_tokens": f1_tokens * 100,   # Return as percentage
        "precision_tokens": avg_precision * 100,
        "recall_tokens": avg_recall * 100
    }


In [ ]:
import torch
from torch.utils.data import DataLoader

def evaluate_model(model, dataloader, tokenizer):
    model.eval()  # Set the model to evaluation mode
    all_predictions = []
    all_labels = []

    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            # Move input data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            # Generate predictions
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=64
            )
            predictions = outputs.cpu().numpy()  # Move predictions to CPU
            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())  # Move labels to CPU

    # Compute metrics
    metrics = compute_metrics(all_predictions, all_labels)
    return metrics


In [ ]:
from torch.utils.data import DataLoader

# Convert the test dataset to a PyTorch DataLoader
test_loader = DataLoader(tokenized_dataset_test, batch_size=8)


In [ ]:
# Evaluate the model on the test set
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


Test Set Metrics: {'exact_match': 40.52953156822811}


# **INFERENCE**

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer
model_path = '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'Using device: {device}')


Using device: cuda
Using device: cuda


In [ ]:
def extract_locations(summary):
    # Prepare the input text with improved prompt
    input_text = f"Extract location hierarchy from incident: {summary}\nFormat: state, district, block, village (comma-separated, in order). Omit missing levels."
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).input_ids.to(device)  # Move input_ids to the device

    # Generate predictions
    outputs = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)

    # Decode the output
    predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_text

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

# Google Maps API key
API_KEY = userdata.get('googlemapsAPI')
GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"


# Updated function to get location details including latitude and longitude
def get_location_details(locations):
    """Given a list of location names, constructs a query, calls the Google Geocoding API,
    and returns state, district, subdistrict, town/village, and latitude/longitude of the most specific level."""
    #query = ', '.join(locations)
    params = {
        'address': locations,
        'key': API_KEY,
        'components': 'country:IN'
    }
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        print(f"Geocoding API error: {data['status']}")
        return None

    # Initialize components
    state = district = subdistrict = town_village = None
    latitude = longitude = None
    found_level = None  # Keep track of the most specific level found

    # Desired levels in order of specificity
    levels = ['locality', 'administrative_area_level_3', 'administrative_area_level_2']

    # Iterate over results to find the most specific level
    for result in data.get('results', []):
        temp_state = temp_district = temp_subdistrict = temp_town_village = None
        address_components = result['address_components']

        # Map address components
        for component in address_components:
            types = component['types']
            if 'administrative_area_level_1' in types:
                temp_state = component['long_name']
            elif 'administrative_area_level_2' in types:
                temp_district = component['long_name']
            elif 'administrative_area_level_3' in types:
                temp_subdistrict = component['long_name']
            elif 'locality' in types:
                temp_town_village = component['long_name']
            elif 'sublocality' in types and not temp_town_village:
                temp_town_village = component['long_name']

        # Determine the most specific level in this result
        if temp_town_village and found_level not in ['town_village']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = temp_town_village
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'town_village'
        elif temp_subdistrict and found_level not in ['town_village', 'subdistrict']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'subdistrict'
        elif temp_district and found_level not in ['town_village', 'subdistrict', 'district']:
            state = temp_state
            district = temp_district
            subdistrict = None
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'district'

        # Break the loop if the most specific level is found
        if found_level == 'town_village':
            break

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village,
        'latitude': latitude,
        'longitude': longitude
    }

In [ ]:
from sklearn.model_selection import train_test_split

# Load the dataset from the URL
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/satp_location_matched_lat_long.csv"

data = pd.read_csv(url, dtype=str)
# Split the dataframe into train, validation, and test sets (80% train, 10% val, 10% test)
train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

# Display the shapes of the split dataframes to verify
print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

Train set shape: (7854, 11)
Validation set shape: (982, 11)
Test set shape: (982, 11)


In [ ]:

# Load the incident data
satp_data = test_df.copy()

satp_data['extracted_state'] = None
satp_data['extracted_district'] = None
satp_data['extracted_subdistrict'] = None
satp_data['extracted_town_village'] = None
satp_data['extracted_latitude'] = None
satp_data['extracted_longitude'] = None



In [ ]:

# Process each summary in the data
for index, row in satp_data.iterrows():
    summary = row['incident_summary']  # Adjust column name if necessary
    # locations = row['Matched_Location']
    locations = extract_locations(summary)
    satp_data.at[index, 'extracted_locations'] = locations
    if locations:
        location_details = get_location_details(locations)
        if location_details:
            # Update the DataFrame with location details
            satp_data.at[index, 'extracted_state'] = location_details.get('state')
            satp_data.at[index, 'extracted_district'] = location_details.get('district')
            satp_data.at[index, 'extracted_subdistrict'] = location_details.get('subdistrict')
            satp_data.at[index, 'extracted_town_village'] = location_details.get('town_village')
            satp_data.at[index, 'extracted_latitude'] = location_details.get('latitude')
            satp_data.at[index, 'extracted_longitude'] = location_details.get('longitude')
    else:
        print(f"No locations found in summary at index {index}")


Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS


In [ ]:
# Define the path to save the CSV file
output_path = '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/satp_data_with_extracted_locations_lat_long.csv'

# Save the DataFrame to CSV
satp_data.to_csv(output_path, index=False)

print(f"DataFrame saved successfully to {output_path}")

DataFrame saved successfully to /content/drive/MyDrive/SATP_data/location_context_extraction_exp/satp_data_with_extracted_locations_lat_long.csv


# Task
Analyze the accuracy of predicted latitude and longitude values against actual values in a dataframe by calculating geographical distances and relevant metrics.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

# Load the DataFrame from the saved CSV file
input_path = '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/satp_data_with_extracted_locations_lat_long.csv'
satp_data = pd.read_csv(input_path)

# Display the first few rows to verify the data
display(satp_data.head())

,incident_number,state,district,block,village_name,other_areas,constituency,incident_summary,Matched_Location,latitude,longitude,extracted_state,extracted_district,extracted_subdistrict,extracted_town_village,extracted_latitude,extracted_longitude,extracted_locations
0,1406200901,Odisha,Kandhmal,Daringbadi,Sonepur,NaN,G. Udayagiri,"Four persons, including the husbands of two el...","kandhmal, daringbadi, sonepur",83.8950,20.8480,Odisha,Southern Division,Kandhamal,Sonpur,19.781595,84.024673,"kandhmal, sonepur"
1,204131401,Bihar,Lakhisarai,NaN,NaN,NaN,NaN,CPI-Maoist cadres blew off a part of the railw...,lakhisarai,NaN,NaN,Bihar,Patna Division,Patna,Patna,25.594095,85.137564,"lakhisarai, howrah, patna"
2,1412021101,Odisha,Koraput,Baipariguda,Mathapada,Kalia Attalla,Kotpad (st),The CPI-Maoist cadres set ablaze two cell phon...,"koraput, baipariguda, mathapada, kalia attalla",82.3072,18.7858,Odisha,Southern Division,Koraput,Boipariguda,18.751105,82.433295,"koraput, baipariguda, kalia attalla"
3,1409041401,Odisha,Balangir,Khaparkhol,Khuripani,Saapmund in the Gandharmardan reserve forest,Patnagarh,"In an anti-Maoist operation, the 189 Battalion...","odisha, balangir, khuripani",82.8627,20.7504,Odisha,Northern Division,Balangir,Khuripani,20.821604,82.829055,"odisha, balangir, khuripani, saapmund"
4,307011001,Chhattisgarh,Narayanpur,NaN,NaN,NaN,NaN,Three civilians were reportedly killed by Maoi...,narayanpur,NaN,NaN,Chhattisgarh,Bastar Division,Narayanpur,Narayanpur,19.719557,81.247197,narayanpur


## Data preparation

### Subtask:
Ensure both actual and predicted latitude and longitude columns are in a suitable format (e.g., numeric). Handle any missing or invalid values.


**Reasoning**:
Convert latitude and longitude columns to numeric, handle missing values by dropping rows with any missing values in the relevant columns, and verify the result.



In [6]:
# Interchange the latitude and longitude columns
satp_data['latitude'], satp_data['longitude'] = satp_data['longitude'], satp_data['latitude']

# Display the first few rows to verify the change
display(satp_data.head())

,incident_number,state,district,block,village_name,other_areas,constituency,incident_summary,Matched_Location,latitude,longitude,extracted_state,extracted_district,extracted_subdistrict,extracted_town_village,extracted_latitude,extracted_longitude,extracted_locations
0,1406200901,Odisha,Kandhmal,Daringbadi,Sonepur,NaN,G. Udayagiri,"Four persons, including the husbands of two el...","kandhmal, daringbadi, sonepur",20.8480,83.8950,Odisha,Southern Division,Kandhamal,Sonpur,19.781595,84.024673,"kandhmal, sonepur"
1,204131401,Bihar,Lakhisarai,NaN,NaN,NaN,NaN,CPI-Maoist cadres blew off a part of the railw...,lakhisarai,NaN,NaN,Bihar,Patna Division,Patna,Patna,25.594095,85.137564,"lakhisarai, howrah, patna"
2,1412021101,Odisha,Koraput,Baipariguda,Mathapada,Kalia Attalla,Kotpad (st),The CPI-Maoist cadres set ablaze two cell phon...,"koraput, baipariguda, mathapada, kalia attalla",18.7858,82.3072,Odisha,Southern Division,Koraput,Boipariguda,18.751105,82.433295,"koraput, baipariguda, kalia attalla"
3,1409041401,Odisha,Balangir,Khaparkhol,Khuripani,Saapmund in the Gandharmardan reserve forest,Patnagarh,"In an anti-Maoist operation, the 189 Battalion...","odisha, balangir, khuripani",20.7504,82.8627,Odisha,Northern Division,Balangir,Khuripani,20.821604,82.829055,"odisha, balangir, khuripani, saapmund"
4,307011001,Chhattisgarh,Narayanpur,NaN,NaN,NaN,NaN,Three civilians were reportedly killed by Maoi...,narayanpur,NaN,NaN,Chhattisgarh,Bastar Division,Narayanpur,Narayanpur,19.719557,81.247197,narayanpur


In [8]:
# Convert latitude and longitude columns to numeric, coercing errors
satp_data['latitude'] = pd.to_numeric(satp_data['latitude'], errors='coerce')
satp_data['longitude'] = pd.to_numeric(satp_data['longitude'], errors='coerce')
satp_data['extracted_latitude'] = pd.to_numeric(satp_data['extracted_latitude'], errors='coerce')
satp_data['extracted_longitude'] = pd.to_numeric(satp_data['extracted_longitude'], errors='coerce')

# Identify and count missing values before dropping
print("Missing values before dropping:")
print(satp_data[['latitude', 'longitude', 'extracted_latitude', 'extracted_longitude']].isnull().sum())

# Drop rows with missing values in any of the relevant coordinate columns
satp_data.dropna(subset=['latitude', 'longitude', 'extracted_latitude', 'extracted_longitude'], inplace=True)

# Verify that there are no remaining missing values
print("\nMissing values after dropping:")
print(satp_data[['latitude', 'longitude', 'extracted_latitude', 'extracted_longitude']].isnull().sum())

# Display the shape of the DataFrame after dropping rows
print(f"\nShape of DataFrame after dropping rows: {satp_data.shape}")

Missing values before dropping:
latitude               0
longitude              0
extracted_latitude     0
extracted_longitude    0
dtype: int64

Missing values after dropping:
latitude               0
longitude              0
extracted_latitude     0
extracted_longitude    0
dtype: int64

Shape of DataFrame after dropping rows: (725, 18)


## Distance calculation

### Subtask:
Calculate the geographical distance (e.g., Haversine distance) between the actual and predicted coordinates for each incident.


**Reasoning**:
Calculate the geographical distance (Haversine distance) between the actual and predicted coordinates for each incident and store it in a new column.



In [ ]:
!pip install haversine

In [ ]:
from haversine import haversine, Unit

# Define a function to calculate the Haversine distance for a row
def calculate_distance(row):
    actual_coords = (row['latitude'], row['longitude'])
    predicted_coords = (row['extracted_latitude'], row['extracted_longitude'])
    # Check if either coordinate pair is missing (though dropna should handle this)
    if pd.isna(actual_coords[0]) or pd.isna(actual_coords[1]) or pd.isna(predicted_coords[0]) or pd.isna(predicted_coords[1]):
        return None
    return haversine(actual_coords, predicted_coords, unit=Unit.KILOMETERS)

# Apply the function to each row to create the 'distance_km' column
satp_data['distance_km'] = satp_data.apply(calculate_distance, axis=1)

# Display the first few rows with the new column
display(satp_data.head())

,incident_number,state,district,block,village_name,other_areas,constituency,incident_summary,Matched_Location,latitude,longitude,extracted_state,extracted_district,extracted_subdistrict,extracted_town_village,extracted_latitude,extracted_longitude,extracted_locations,distance_km
8066,1406200901,Odisha,Kandhmal,Daringbadi,Sonepur,NaN,G. Udayagiri,"Four persons, including the husbands of two el...","kandhmal, daringbadi, sonepur",20.848000,83.895000,Odisha,Southern Division,Kandhamal,Sonpur,19.781595,84.024673,"kandhmal, sonepur",119.347457
8712,1412021101,Odisha,Koraput,Baipariguda,Mathapada,Kalia Attalla,Kotpad (st),The CPI-Maoist cadres set ablaze two cell phon...,"koraput, baipariguda, mathapada, kalia attalla",18.785800,82.307200,Odisha,Southern Division,Koraput,Boipariguda,18.751105,82.433295,"koraput, baipariguda, kalia attalla",13.824810
8365,1409041401,Odisha,Balangir,Khaparkhol,Khuripani,Saapmund in the Gandharmardan reserve forest,Patnagarh,"In an anti-Maoist operation, the 189 Battalion...","odisha, balangir, khuripani",20.750400,82.862700,Odisha,Northern Division,Balangir,Khuripani,20.821604,82.829055,"odisha, balangir, khuripani, saapmund",8.655664
6761,904120901,Karnataka,Chikmagalur,Koppa,Megunda,Baisigadde,Sringeri,Police arrested three sympathizers of the CPI-...,"chikmagalur, koppa, megunda, baisigadde, sringeri",13.337800,75.322300,Karnataka,Mysore Division,Chikkamagaluru,Megunda,13.337837,75.322268,"chikmagalur, kachige, megunda revenue division",0.005360
1945,201221401,Bihar,Patna,NaN,NaN,Patna Medical College,NaN,The Patna city Police of Bihar arrested a CPI-...,"bihar, patna, patna medical college",25.621017,85.160416,Bihar,Patna Division,Patna,Patna,25.594095,85.137564,"bihar, patna",3.769916


## Analysis and scoring

### Subtask:
Analyze the distribution of distances. You can calculate metrics like the mean, median, or standard deviation of the distances. You could also define thresholds for "close" predictions and calculate the percentage of predictions within those thresholds.


**Reasoning**:
Calculate and print the mean, median, and standard deviation of the distance_km column, and then calculate and print the percentage of incidents within defined distance thresholds.



In [ ]:
import numpy as np

# Calculate mean, median, and standard deviation of distance_km
mean_distance = satp_data['distance_km'].mean()
median_distance = satp_data['distance_km'].median()
std_distance = satp_data['distance_km'].std()

# Print the calculated metrics
print(f"Mean distance: {mean_distance:.2f} km")
print(f"Median distance: {median_distance:.2f} km")
print(f"Standard deviation of distance: {std_distance:.2f} km")

# Define distance thresholds
thresholds = [1, 5, 10, 50, 100]

# Calculate and print the percentage of incidents within each threshold
print("\nPercentage of incidents within distance thresholds:")
for threshold in thresholds:
    within_threshold = satp_data[satp_data['distance_km'] <= threshold].shape[0]
    percentage = (within_threshold / satp_data.shape[0]) * 100
    print(f"  <= {threshold} km: {percentage:.2f}%")

Mean distance: 82.09 km
Median distance: 18.62 km
Standard deviation of distance: 249.91 km

Percentage of incidents within distance thresholds:
  <= 1 km: 32.97%
  <= 5 km: 38.62%
  <= 10 km: 42.90%
  <= 50 km: 68.41%
  <= 100 km: 81.52%


## Stratified Analysis by Location Level

### Subtask:
Separate the accuracy metrics by the location level of the original coordinates (village, block, district, or state-level centroids) to better understand model performance at different granularities.


**Reasoning**:
Determine the most specific location level available for each row based on which fields are populated (village_name > block > district > state), then calculate separate distance metrics for each level. This allows for fairer evaluation since expectations for accuracy should differ based on the granularity of the original coordinates.


In [ ]:
# Determine the original location level for each row
# Priority: village_name > block > district > state
def determine_location_level(row):
    """Determine the most specific location level available for this row."""
    if pd.notna(row.get('village_name')) and str(row.get('village_name')).strip() != '':
        return 'village'
    elif pd.notna(row.get('block')) and str(row.get('block')).strip() != '':
        return 'block'
    elif pd.notna(row.get('district')) and str(row.get('district')).strip() != '':
        return 'district'
    elif pd.notna(row.get('state')) and str(row.get('state')).strip() != '':
        return 'state'
    else:
        return 'unknown'

# Add location level column
satp_data['original_location_level'] = satp_data.apply(determine_location_level, axis=1)

# Display the distribution of location levels
print("Distribution of original location levels:")
print(satp_data['original_location_level'].value_counts())
print(f"\nTotal rows: {len(satp_data)}")


In [ ]:
# Calculate metrics for each location level
location_levels = ['village', 'block', 'district', 'state', 'unknown']
results_by_level = {}

for level in location_levels:
    level_data = satp_data[satp_data['original_location_level'] == level]
    
    if len(level_data) == 0:
        continue
    
    results_by_level[level] = {
        'count': len(level_data),
        'mean_distance': level_data['distance_km'].mean(),
        'median_distance': level_data['distance_km'].median(),
        'std_distance': level_data['distance_km'].std(),
        'min_distance': level_data['distance_km'].min(),
        'max_distance': level_data['distance_km'].max(),
    }
    
    # Calculate percentages within thresholds
    thresholds = [1, 5, 10, 50, 100]
    for threshold in thresholds:
        within_threshold = len(level_data[level_data['distance_km'] <= threshold])
        percentage = (within_threshold / len(level_data)) * 100
        results_by_level[level][f'pct_within_{threshold}km'] = percentage

# Create a summary dataframe
summary_df = pd.DataFrame(results_by_level).T
summary_df = summary_df.round(2)

print("Metrics by Original Location Level:")
print("=" * 80)
display(summary_df)


In [ ]:
# Print detailed metrics for each level
print("\nDetailed Metrics by Location Level:")
print("=" * 80)

for level in ['village', 'block', 'district', 'state']:
    if level not in results_by_level:
        continue
    
    level_data = satp_data[satp_data['original_location_level'] == level]
    metrics = results_by_level[level]
    
    print(f"\n{level.upper()}-LEVEL COORDINATES (n={metrics['count']}):")
    print(f"  Mean distance: {metrics['mean_distance']:.2f} km")
    print(f"  Median distance: {metrics['median_distance']:.2f} km")
    print(f"  Standard deviation: {metrics['std_distance']:.2f} km")
    print(f"  Range: {metrics['min_distance']:.2f} - {metrics['max_distance']:.2f} km")
    print(f"\n  Percentage within thresholds:")
    thresholds = [1, 5, 10, 50, 100]
    for threshold in thresholds:
        print(f"    <= {threshold} km: {metrics[f'pct_within_{threshold}km']:.2f}%")


In [ ]:
# Optional: Create visualizations comparing distributions across levels
import matplotlib.pyplot as plt

# Box plot of distances by location level
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot
levels_to_plot = [level for level in ['village', 'block', 'district', 'state'] 
                  if level in results_by_level]
data_to_plot = [satp_data[satp_data['original_location_level'] == level]['distance_km'].values 
                for level in levels_to_plot]

axes[0].boxplot(data_to_plot, labels=levels_to_plot)
axes[0].set_ylabel('Distance (km)')
axes[0].set_xlabel('Original Location Level')
axes[0].set_title('Distribution of Distance Errors by Location Level')
axes[0].set_yscale('log')  # Use log scale for better visualization
axes[0].grid(True, alpha=0.3)

# Bar chart of mean distances
means = [results_by_level[level]['mean_distance'] for level in levels_to_plot]
axes[1].bar(levels_to_plot, means)
axes[1].set_ylabel('Mean Distance (km)')
axes[1].set_xlabel('Original Location Level')
axes[1].set_title('Mean Distance Error by Location Level')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nNote: Log scale used for box plot to better visualize differences across levels.")
